# 1. Install necessary Libraries

In this part, we'll begin by installing necessary libraries needed for running our computer vision training and testing scripts

In [3]:
%pip install torch torchvision torchaudio
%pip install transformers scikit-learn pillow pandas numpy huggingface_hub ipywidgets opencv-python

Looking in indexes: https://download.pytorch.org/whl/cu128
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


  Obtaining dependency information for ipywidgets from https://files.pythonhosted.org/packages/56/6d/0d9848617b9f753b87f214f1c682592f7ca42de085f564352f10f0843026/ipywidgets-8.1.8-py3-none-any.whl.metadata
  Obtaining dependency information for widgetsnbextension~=4.0.14 from https://files.pythonhosted.org/packages/3f/0e/fa3b193432cfc60c93b42f3be03365f5f909d2b3ea410295cf36df739e31/widgetsnbextension-4.0.15-py3-none-any.whl.metadata
  Obtaining dependency information for jupyterlab_widgets~=3.0.15 from https://files.pythonhosted.org/packages/ab/b5/36c712098e6191d1b4e349304ef73a8d06aed77e56ceaac8c0a306c7bda1/jupyterlab_widgets-3.0.16-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/139.8 kB ? eta -:--:--
   -- ------------------------------------- 10.2/139.8 kB ? eta -:--:--
   -------- ------------------------------ 30.7/139.8 kB 435.7 kB/s eta 0:00:01
   ----------------- --------------------- 61.4/139.8 kB 465.5 kB/s eta 0:00:01
   -----------------------------


[notice] A new release of pip is available: 23.2.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# 2. Huggingface login via UI
This part is optional as you're still able to access huggingface API without authentication, just that there's stricter rate limiting. If you have not faced any rate limiting issues can comment out or delete this cell

In [4]:
from huggingface_hub import notebook_login
notebook_login()

# 3. CLAHE Data Preprocessing function
In this part, we'll write Contrast Limited Adaptive Histogram Equalization (CLAHE) preprocessing function logic. Since this data preprocessing is added to all dataset (training/val/test) and its not part of data augmentation, we'll separate them for now

In [ ]:
import cv2
import numpy as np
from PIL import Image

def apply_clahe(image):
    image = np.array(image)

    lab = cv2.cvtColor(image, cv2.COLOR_RGB2LAB)
    l, a, b = cv2.split(lab)

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8,8))
    cl = clahe.apply(l)

    merged = cv2.merge((cl, a, b))
    enhanced = cv2.cvtColor(merged, cv2.COLOR_LAB2RGB)

    return Image.fromarray(enhanced)

# 4. Custom Dataset Loader
Since we're using a custom dataset, we will need to write a custom dataset loader to pass our image data to the model


In [5]:
from torch.utils.data import Dataset
from PIL import Image
import os

class MedicalDatasetLoader(Dataset):
    def __init__(self, df, img_dir, transform=None, use_clahe=True):
        self.data = df # Differentiate between train_df, val_df and test_df
        self.img_dir = img_dir # Image Directory Path
        self.transform = transform # Data augmentation
        self.use_clahe = use_clahe # Apply CLAHE data preprocessing

    def __len__(self):
        return len(self.data) # Calculate the number of rows of the dataset

    def __getitem__(self, idx):
        img_name = self.data.iloc[idx]['Image'] # Get the image name from the csv header. Used for path later
        label = int(self.data.iloc[idx]['Label']) # Get the Label value from csv header

        img_path = os.path.join(self.img_dir, img_name) # Combine the root image dir with the image name to get the full individual image path
        image = Image.open(img_path).convert("RGB")

        if self.use_clahe: # Implement CLAHE for training set, disable for val and testing set
            image = apply_clahe(image)

        if self.transform: # Implement Data Augmentation
            image = self.transform(image)

        return image, label

# 5. Setup MPS Device
Here in this part, we check if PyTorch detects our GPU MPS device on macOS

In [6]:
import torch

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu") # Check if MPS is detected by pytorch, fallback to CPU otherwise
print("Using device:", device)

Cuda available:  True


# 6. Setup Model
In this part, we will load the SwinV2 (Swin Transformer V2) model from HuggingFace. The model is pretrained on ImageNet-1K at 256x256 resolution but we use 224x224 for cross-model consistency.

In [ ]:
from transformers import Swinv2ForImageClassification, AutoImageProcessor

# Load Microsoft SwinV2 model
model = Swinv2ForImageClassification.from_pretrained(
    'microsoft/swinv2-base-patch4-window8-256',
    num_labels=5,
    ignore_mismatched_sizes=True
).to(device)

processor = AutoImageProcessor.from_pretrained('microsoft/swinv2-base-patch4-window8-256')

# 7. Data Preprocessing
In this part, we will perform very basic data preprocessing on our dataset, then split our dataset into training and validation data. Since this is a baseline model, we'll run it without any data augmentation for now, feel free to add any augmentation like rotation, colour jitter and crop via transforms library from torchvision.

In [7]:
from torchvision import transforms
from sklearn.model_selection import train_test_split
from torch.utils.data import DataLoader
import pandas as pd

# Data augmentation
train_transform = transforms.Compose([# For training dataset it is recommended to test around with different augmentations like rotation, flip and colour change to ensure more diverse dataset during training so that our model can see more amount of images and improve its generalization. CLAHE has been added beforehand so no need add here. As this is a baseline model no augmentation is added for now.
    transforms.Resize((224, 224)), # Ensure all dataset image is fixed to 224x224 pixels
    transforms.ToTensor(), # Convert the image pixels into matrix format (Tensor is pytorch matrix format)
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std) # We use the mean and std of the model based on the processor class earlier
])

val_transform = transforms.Compose([# For validation & testing dataset, don't add any data augmentation as we do not need diverse dataset when testing model performance, that's training job for that. In both dataset we just have resize the images, convert to tensor matrix and normalization. DON'T add any other here.
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=processor.image_mean, std=processor.image_std) # Same as training normalization above
])

# Normal train_test_split
df = pd.read_csv('dataset/labels.csv') # Read dataset from our CSV file

# Step 1: Train+Val / Test
train_val_df, test_df = train_test_split(
    df,
    test_size=0.2,
    stratify=df['Label'],
    random_state=42
)

# Step 2: Train / Val
train_df, val_df = train_test_split(
    train_val_df,
    test_size=0.2,
    stratify=train_val_df['Label'],
    random_state=42
)
# We implement the data augmentation transform (train_transform) version for our training dataset, and enabling CLAHE inside the training dataset
train_dataset = MedicalDatasetLoader(train_df, "dataset/image/", train_transform, use_clahe=True)

# We implement the non-augmented data transform (val_transform) version for both validation and testing dataset, and disabling CLAHE for both
val_dataset = MedicalDatasetLoader(val_df, "dataset/image/", val_transform, use_clahe=False)
test_dataset = MedicalDatasetLoader(test_df, "dataset/image", val_transform, use_clahe=False)

# DataLoader here is to load our training dataset and split it into smaller batches known as mini-batch. This is because our GPU is not able to fit in all images from training/val/test dataset at the same time. So, for each iteration(epoch), we split the dataset into per (batch_size, i.e. 16), then loop through every batch size until we completed the entire dataset. (Exp: 3200 training dataset, every epoch we loop through every 16 images, and repeat this for 3200/16 = 200 times then we move to next epoch)
train_loader = DataLoader(
    train_dataset,
    batch_size=16,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=16,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False
)

print(f"Train: {len(train_df)}")
print(f"Val: {len(val_df)}")
print(f"Test: {len(test_df)}")

# 8. Hyperparameters tuning
We set up our class weights, epochs number, early stopping mechanism, optimizer and learning rate scheduler to optimize our model performance and provide better generalization through reducing overfitting

In [9]:
import torch.nn as nn
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from torch.amp import GradScaler
from sklearn.utils.class_weight import compute_class_weight

class_weights = compute_class_weight('balanced', classes=np.unique(train_df['Label']), y=train_df['Label']) # Since we have very imbalanced data classes (i.e. 3000 class 0 and 200 class 3), we need to give more focus towards the minor class and less focus towards the major class.
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device) # Convert the class_weights value into pytorch matrix, and store in GPU

num_epochs = 20
early_stopping_patience = 5
epochs_no_improvement = 0
best_val_loss = float('inf')
early_stopping = False

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5, weight_decay=1e-4)

# Learning Rate Scheduler: Adjust the lr value every epoch to improve convergence. Higher lr at the beginning since the loss curve is steep, while lower lr at the end since the loss curve is flat, and we don't want to overshoot past minimum point
scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs, eta_min=1e-6)

# Loss Function
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

# Scaler by pytorch. Here we use a mixed precision decimals which is FP32(32 decimals) and FP16(16 decimals). For more important calculation like calculating loss, updating weights and bias we use 32 decimals, whereas for calculating gradients we use 16 decimals. The goal here is to ensure efficient memory usage to prevent Cuda out of memory.
# The scaler here consists of a scale value (i.e. 1000) to increase the value before rounding off from FP32 -> FP16 to prevent rounding off to 0 issue. Likewise when we downscale the value from FP16 to FP32, we use the same scale value as well.
# The scale value is handled internally by the Pytorch Library, and we don't have to do anything to it. Just remember to update the scale value with scaler.update() to adjust the scale value after each epoch iteration to ensure optimized value
scaler = GradScaler()

# 9. Experiment Logging Class
In this part, we'll create a log class tha can help us to logs our hyperparameter lists as well as result metrics.

In [10]:
import json
from datetime import datetime
import os

class ExperimentTracker:
    def __init__(self, base_dir="experiments"):
        self.base_dir = base_dir
        os.makedirs(base_dir, exist_ok=True)

        timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
        self.exp_dir = os.path.join(base_dir, f"exp_{timestamp}")
        os.makedirs(self.exp_dir)

        self.epoch_metrics = []
        self.final_metrics = {}
        self.config = {}

    # ---------------- CONFIG ----------------
    def log_config(self, config):
        self.config = config
        with open(os.path.join(self.exp_dir, "config.json"), "w") as f:
            json.dump(config, f, indent=4)

    # ---------------- PER EPOCH ----------------
    def log_epoch(self, epoch, train_loss, val_loss, train_acc, val_acc):
        self.epoch_metrics.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "val_loss": val_loss,
            "train_acc": train_acc,
            "val_acc": val_acc
        })

    # ---------------- FINAL METRICS ----------------
    def log_final_metrics(self, split, acc, prec, rec, f1, roc_auc, cm):
        if "final_metrics" not in self.__dict__ or not isinstance(self.final_metrics, dict):
            self.final_metrics = {}

        self.final_metrics[split] = {
            "accuracy": acc,
            "precision": prec,
            "recall": rec,
            "f1_score": f1,
            "roc_auc_score": roc_auc,
            "confusion_matrix": cm.tolist()
        }

    # ---------------- SAVE ----------------
    def save_all(self):
        with open(os.path.join(self.exp_dir, "metrics.json"), "w") as f:
            json.dump({
                "config": self.config,
                "epoch_metrics": self.epoch_metrics,
                "final_metrics": self.final_metrics
            }, f, indent=4)

    def save_model(self, model, name="best_model.pth"):
        torch.save(model.state_dict(), os.path.join(self.exp_dir, name))

# 10. Training and Validation Script
In this part, we train and test our model and finally save the final weights of the model

In [9]:
CONFIG = {
    "learning_rate": 5e-5,
    "weight_decay": 1e-4,
    "epochs": num_epochs,
    "batch_size": 16,
    "optimizer": "AdamW",
    "scheduler": "CosineAnnealingLR"
}

tracker = ExperimentTracker()
tracker.log_config(CONFIG)

import torch.nn.functional as F
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,roc_auc_score)

for epoch in range(num_epochs):
    model.train()
    train_loss, train_correct, train_total = 0.0, 0, 0

    # ---------------- TRAINING ----------------
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device) # Put the image pixels matrix and labels (classification class) into our GPU

        optimizer.zero_grad() # Put the image pixels matrix and labels (classification class) into our GPU

        outputs = model(pixel_values=inputs)
        loss = criterion(outputs.logits, labels) # Calculating loss

        loss.backward() # Calculating gradients via backpropagation
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0) # Weights and Bias value maximum 1.0 to prevent exploding gradient -> ∞
        optimizer.step() # Skips bad gradient like gradient with 0 or NaN values to avoid corruption of weights and bias

        train_loss += loss.item() * inputs.size(0) # Store loss value
        _, predicted = torch.max(outputs.logits, 1)
        train_total += labels.size(0)
        train_correct += (predicted == labels).sum().item()

    # ---------------- VALIDATION ----------------
    model.eval() # Testing phase
    val_loss, val_correct, val_total = 0.0, 0, 0
    all_preds, all_labels, all_probs = [], [], []

    with torch.no_grad(): # Since we're not calculating gradient using backpropagation when testing, we don't need to store gradient value which saves space
        # Everything from here is similar to training phase above
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(pixel_values=inputs)
            loss = criterion(outputs.logits, labels)

            probs = F.softmax(outputs.logits, dim=1) # Get probability output from the final layer through softmax function

            val_loss += loss.item() * inputs.size(0)
            _, predicted = torch.max(outputs.logits, 1)

            val_total += inputs.size(0)
            val_correct += (predicted == labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.detach().cpu().numpy())

    # ---------------- METRICS ----------------
    epoch_train_loss = train_loss / train_total
    epoch_train_acc = train_correct / train_total
    epoch_val_loss = val_loss / val_total
    epoch_val_acc = val_correct / val_total

    f1 = f1_score(all_labels, all_preds, average='macro')
    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    cm = confusion_matrix(all_labels, all_preds)
    try:
        roc_auc = roc_auc_score(
            np.array(all_labels),
            np.array(all_probs),
            multi_class='ovr'
        )
    except ValueError:
        roc_auc = 0.0

    scheduler.step() # Adjust the learning rate value through our scheduler function in cell 7

    # ---------------- TRACKING ----------------
    tracker.log_epoch(epoch, epoch_train_loss, epoch_val_loss, epoch_train_acc, epoch_val_acc)

    if epoch_val_loss < best_val_loss:
        best_val_loss = epoch_val_loss
        epochs_no_improvement = 0
        tracker.save_model(model)
    else:
        epochs_no_improvement += 1
        if epochs_no_improvement >= early_stopping_patience:
            print('Early stopping')
            break

    print(f"Epoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f}")
    print(f"Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

# 🔥 LOG VALIDATION
tracker.log_final_metrics("validation", acc, prec, rec, f1, roc_auc, cm)
tracker.save_all()

Epoch [1/20]
Train Loss: 1.4421 | Train Accuracy: 59.38%
Test Loss: 1.3125 | Test Accuracy: 69.13%
Epoch [2/20]
Train Loss: 1.2626 | Train Accuracy: 70.79%
Test Loss: 1.2137 | Test Accuracy: 76.62%
Epoch [3/20]
Train Loss: 1.1685 | Train Accuracy: 74.51%
Test Loss: 1.1759 | Test Accuracy: 77.43%
Epoch [4/20]
Train Loss: 1.0967 | Train Accuracy: 79.63%
Test Loss: 1.2183 | Test Accuracy: 71.96%
Epoch [5/20]
Train Loss: 1.0296 | Train Accuracy: 82.51%
Test Loss: 1.1405 | Test Accuracy: 78.64%
Epoch [6/20]
Train Loss: 0.9728 | Train Accuracy: 86.13%
Test Loss: 1.1334 | Test Accuracy: 78.14%
Epoch [7/20]
Train Loss: 0.9170 | Train Accuracy: 87.55%
Test Loss: 1.1302 | Test Accuracy: 77.94%
Epoch [8/20]
Train Loss: 0.8711 | Train Accuracy: 90.21%
Test Loss: 1.1964 | Test Accuracy: 83.30%
Epoch [9/20]
Train Loss: 0.8246 | Train Accuracy: 92.41%
Test Loss: 1.1569 | Test Accuracy: 79.66%
Epoch [10/20]
Train Loss: 0.7817 | Train Accuracy: 93.80%
Test Loss: 1.2059 | Test Accuracy: 81.38%
Epoch [11

# 11. Testing Script & Metrics Visualization
Lastly, in the final parts we'll measure our model performance based on the following metrics:
- Accuracy: Calculate the ratio of correctly predicted labels
- Precision: Calculate how many positive predictions are truly positive (Minimize false positives)
- Recall: Calculate how many postivive labels are predicted (Minimize false negatives)
- F1-Score: Determine the mean of precision and recall combined
- ROC-AUC Score: Determine how confident our model is at classifying the classes (0.5 - Random Guessing, ~1.0: Almost 100% accurate)
- Confusion Matrix: Determine the total True Positive, False Positives, True Negatives and False Negatives that our model made

In [10]:
def evaluate_and_log(model, test_loader, device, tracker):
    model.eval()

    all_preds = []
    all_labels = []
    all_probs = []

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(pixel_values=inputs)

            probs = F.softmax(outputs.logits, dim=1)
            _, predicted = torch.max(outputs.logits, 1)

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            all_probs.extend(probs.cpu().numpy())

    acc = accuracy_score(all_labels, all_preds)
    prec = precision_score(all_labels, all_preds, average='macro', zero_division=0)
    rec = recall_score(all_labels, all_preds, average='macro', zero_division=0)
    f1 = f1_score(all_labels, all_preds, average='macro')
    cm = confusion_matrix(all_labels, all_preds)

    try:
        roc_auc = roc_auc_score(
            np.array(all_labels),
            np.array(all_probs),
            multi_class='ovr'
        )
    except ValueError:
        roc_auc = 0.0

    # 🔥 LOG TEST
    tracker.log_final_metrics("test", acc, prec, rec, f1, roc_auc, cm)
    tracker.save_all()

    print("\n📊 TEST RESULTS")
    print(f"Accuracy: {acc * 100:.2f}%")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall: {rec * 100:.2f}%")
    print(f"F1 Score: {f1 * 100:.2f}%")
    print(f"ROC-AUC: {roc_auc * 100:.2f}%")
    print("Confusion Matrix:")
    print(cm)

    return acc, prec, rec, f1, roc_auc, cm

Accuracy: 82.09%
Precision: 70.14%
Recall: 67.99%
F1-Score: 68.97%
Confusion Matrix: 
[[437   9   2   0   0]
 [ 17  58  27   1   2]
 [ 11  24 215  16  11]
 [  0   1  21  15   7]
 [  0   1  19   8  86]]
